# 03 — Execução e pré-processamento

**Entrada:** plano aprovado. **Saída:** dados brutos preservados, nuvem registrada e evidências de controle de campo.

> Nunca altere os dados brutos. Trabalhe em cópias versionadas e registre cada parâmetro de processamento.

## Objetivos de aprendizagem

1. executar a aquisição cinemática com segurança e rastreabilidade;
2. detectar falhas de cobertura, deriva e perda de rastreamento ainda em campo;
3. preservar, inventariar e verificar os dados brutos;
4. aplicar filtragem e amostragem sem destruir detalhes necessários;
5. decidir se é necessária uma campanha complementar;
6. distinguir nuvem, malha, Gaussian Splatting e 2DGS.

## 1. Campanha 1 — aquisição

### Sequência operacional

1. briefing de segurança, autorizações e inspeção do local;
2. identificação do equipamento, operador, configuração e horário;
3. inicialização em área rica em geometria e teste curto;
4. percurso suave, com velocidade estável e loops planejados;
5. marcação de interrupções, eventos, pessoas/objetos móveis e zonas degradadas;
6. encerramento no ponto planejado e verificação imediata;
7. duas cópias verificadas antes de apagar ou reutilizar a mídia.

Não encubra alertas do software. Em caso de perda de rastreamento, retorne a uma região reconhecível conforme o procedimento do fabricante e anote o evento.

In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

diario = pd.DataFrame([
    {'hora': '08:00', 'evento': 'Inspeção e inicialização', 'status': 'OK', 'observacao': ''},
    {'hora': '08:20', 'evento': 'Trecho de teste', 'status': 'A VALIDAR', 'observacao': 'Processar antes da missão'},
    {'hora': '09:00', 'evento': 'Campanha 1', 'status': 'PLANEJADO', 'observacao': ''},
])
diario

## 2. Controle de qualidade em campo

Verifique completude visual, trajetória, continuidade, loops, alinhamento de superfícies repetidas, cores/imagens, duração, tamanho dos arquivos e logs. Inspecione especialmente quinas, escadas, fachadas, áreas com vidro, vegetação e transições interior–exterior.

Critérios de repetição devem vir do plano. Exemplos: lacuna em área obrigatória, loop que não fecha, dupla parede, deslocamento visível entre passagens, log incompleto ou ausência de ponto de verificação.

In [ ]:
check_campo = pd.DataFrame({
    'item': ['arquivos abrem', 'trajetória contínua', 'loop fechado', 'cobertura crítica', 'backup A', 'backup B'],
    'resultado': ['A VERIFICAR'] * 6,
    'evidencia': [''] * 6,
})
check_campo

## 3. Pré-processamento reprodutível

O software do fabricante normalmente estima trajetória e registro SLAM. Exporte também logs/poses quando disponíveis. Em ferramentas abertas, mantenha as etapas separadas: leitura, recorte, remoção de inválidos, filtragem de outliers, redução de densidade, registro complementar e classificação.

A demonstração abaixo cria uma nuvem sintética com plano, fachada e outliers. Ela permite testar o raciocínio sem dados do equipamento.

In [ ]:
rng = np.random.default_rng(100)
n = 4000
piso = np.column_stack([rng.uniform(0, 20, n), rng.uniform(0, 12, n), rng.normal(0, .02, n)])
parede = np.column_stack([rng.normal(0, .02, n//2), rng.uniform(0, 12, n//2), rng.uniform(0, 4, n//2)])
outliers = rng.uniform([-5, -5, -3], [25, 17, 10], size=(120, 3))
pontos = np.vstack([piso, parede, outliers])
pontos.shape

### 3.1 Remoção didática de outliers

A distância média aos vizinhos é uma medida local. O limiar deve ser inspecionado e documentado; filtragem excessiva pode apagar cabos, quinas e vegetação legítima.

In [ ]:
arvore = cKDTree(pontos)
distancias, _ = arvore.query(pontos, k=9)
distancia_media = distancias[:, 1:].mean(axis=1)
limite = np.median(distancia_media) + 4 * np.median(np.abs(distancia_media - np.median(distancia_media)))
mascara = distancia_media <= limite
pontos_filtrados = pontos[mascara]
pd.Series({'originais': len(pontos), 'mantidos': mascara.sum(), 'removidos': (~mascara).sum(), 'limiar_m': limite}).to_frame('valor')

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, dados, titulo in zip(eixos, [pontos, pontos_filtrados], ['Antes', 'Depois']):
    ax.scatter(dados[:,0], dados[:,2], s=1, alpha=.4)
    ax.set(title=titulo, xlabel='X (m)', ylabel='Z (m)')
    ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

### 3.2 Amostragem por voxel

A amostragem abaixo mantém um representante por célula. Use o tamanho do voxel em função do menor detalhe necessário e conserve a nuvem de maior resolução.

In [ ]:
voxel_m = 0.05
indices_voxel = np.floor(pontos_filtrados / voxel_m).astype(np.int64)
_, indices_unicos = np.unique(indices_voxel, axis=0, return_index=True)
pontos_reduzidos = pontos_filtrados[np.sort(indices_unicos)]
print(f'{len(pontos_filtrados):,} -> {len(pontos_reduzidos):,} pontos com voxel de {voxel_m:.2f} m')

## 4. Campanha 2 e fechamento de loop

A campanha complementar deve responder a uma lista objetiva de lacunas. Garanta sobreposição com regiões estáveis da primeira campanha, repita alvos de controle e evite depender apenas de objetos móveis ou vegetação. Depois do registro, avalie se surgiram superfícies duplicadas e se os resíduos nos pontos de verificação pioraram.

## 5. Reconstrução e representações

- **Poisson:** superfície contínua; exige normais consistentes e pode fechar buracos indevidamente.
- **Alpha Shape:** controla concavidades; sensível ao parâmetro e à densidade.
- **Gaussian Splatting:** representação radiométrica voltada à síntese de vistas; não equivale automaticamente a uma superfície métrica.
- **2D Gaussian Splatting:** pode favorecer geometria de superfície, mas ainda precisa de validação independente para uso métrico.

Produza malha/splat somente quando o requisito justificar. Preserve a nuvem registrada como evidência primária.

## 6. Verificação e entrega do módulo

- [ ] brutos preservados e hashes registrados;
- [ ] diário, configuração, logs e trajetória arquivados;
- [ ] cobertura e loops inspecionados;
- [ ] campanha complementar justificada por lacunas;
- [ ] parâmetros e versões do pipeline registrados;
- [ ] contagens antes/depois de cada filtro comparadas;
- [ ] nuvem processada pronta para controle independente.

**Entrega do módulo:** pacote bruto inventariado, relatório de campo e nuvem registrada.

**Próximo módulo:** [04 — Análise](04_Analise.ipynb).